In [1]:
import pandas as pd

train_df = pd.read_csv('/train.csv')

# Question 1

In [2]:
answer_mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
train_df['answer_encoded'] = train_df['answer'].map(answer_mapping)

encoded_label_at_150 = train_df.loc[150, 'answer_encoded']
print(f"The encoded numeric label for the row at index 150 is: {encoded_label_at_150}")

The encoded numeric label for the row at index 150 is: 2


# Question 2

In [3]:
prompt_text = train_df.loc[0, 'prompt']
option_b_text = train_df.loc[0, 'B']

formatted_string = str(prompt_text) + " [SEP] " + str(option_b_text)
length_of_string = len(formatted_string)

print(f"The exact character length of the formatted input string for row 0, Option B is: {length_of_string}")

The exact character length of the formatted input string for row 0, Option B is: 407


# Question 3

In [4]:
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

row_index_0 = train_df.loc[0]
prompt_0 = row_index_0['prompt']

options_0 = [row_index_0['A'], row_index_0['B'], row_index_0['C'], row_index_0['D'], row_index_0['E']]

formatted_inputs_0 = []
for opt in options_0:
    formatted_inputs_0.append(str(prompt_0) + " [SEP] " + str(opt))

tokenized_inputs = tokenizer(
    formatted_inputs_0,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Reshape for a multiple-choice model
input_ids_reshaped = tokenized_inputs['input_ids'].unsqueeze(0)

print(f"Original input_ids shape: {tokenized_inputs['input_ids'].shape}")
print(f"Reshaped input_ids shape: {input_ids_reshaped.shape}")
print(f"The value of the second dimension is: {input_ids_reshaped.shape[1]}")

Original input_ids shape: torch.Size([5, 128])
Reshaped input_ids shape: torch.Size([1, 5, 128])
The value of the second dimension is: 5


# Question 4

In [5]:
all_formatted_inputs = []

# Process the first 16 rows
for i in range(16):
    row = train_df.loc[i]
    prompt = row['prompt']
    options = [row['A'], row['B'], row['C'], row['D'], row['E']]

    for opt in options:
        all_formatted_inputs.append(str(prompt) + " [SEP] " + str(opt))

# Tokenize all formatted inputs
batch_tokenized_inputs = tokenizer(
    all_formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Reshape the input_ids tensor to [16, 5, 128]
input_ids_batch_reshaped = batch_tokenized_inputs['input_ids'].view(16, 5, 128)

print(f"Shape of the final input_ids tensor: {input_ids_batch_reshaped.shape}")

total_token_positions = input_ids_batch_reshaped.numel()
print(f"Total token positions in this tensor: {total_token_positions}")

Shape of the final input_ids tensor: torch.Size([16, 5, 128])
Total token positions in this tensor: 10240


# Question 5

In [6]:
from transformers import AutoModelForMultipleChoice

model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

outputs = model(input_ids=input_ids_reshaped)
logits = outputs.logits

print(f"Shape of the output logits tensor: {logits.shape}")
print(f"Number of logits produced for one question: {logits.shape[1]}")

W0716 10:21:38.094000 20563 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0716 10:21:38.222000 20563 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Shape of the output logits tensor: torch.Size([1, 5])
Number of logits produced for one question: 5


# Question 6

In [7]:
# Get the correct encoded label for row index 0
correct_label_q6 = train_df.loc[0, 'answer_encoded']

# The model expects labels as a tensor
labels = torch.tensor([correct_label_q6])

# Pass the tokenized input and the labels to the model
outputs_with_labels = model(input_ids=input_ids_reshaped, labels=labels)
loss = outputs_with_labels.loss

print(f"Loss tensor: {loss}")
print(f"Number of dimensions of the loss tensor: {loss.ndim}")

Loss tensor: 1.586911678314209
Number of dimensions of the loss tensor: 0


# Question 7

In [8]:
!pip install --upgrade torchao

from peft import LoraConfig, get_peft_model, TaskType

# Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

# Apply LoRA to the model
peft_model = get_peft_model(model, lora_config)

# Count trainable parameters
trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)

print(f"Number of trainable parameters after applying LoRA: {trainable_params}")

Number of trainable parameters after applying LoRA: 295681


# Question 8

In [9]:
from datasets import Dataset
import torch

# Select the first 100 rows from train_df
dataset_df = train_df.head(100)

def preprocess_function(examples):
    all_prompts = []
    all_options = []
    all_labels = []

    for i in range(len(examples['prompt'])):
        prompt = examples['prompt'][i]
        options = [examples[col][i] for col in ['A', 'B', 'C', 'D', 'E']]
        label = examples['answer_encoded'][i]

        # Create 5 formatted strings for each example (prompt + option)
        formatted_choices = [str(prompt) + " [SEP] " + str(opt) for opt in options]

        all_prompts.extend(formatted_choices)
        all_labels.append(label)

    # Tokenize all formatted choices in one go
    tokenized_batch = tokenizer(
        all_prompts,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    # Remove token_type_ids
    if 'token_type_ids' in tokenized_batch:
        del tokenized_batch['token_type_ids']

    num_choices = 5
    batch_size = len(examples['prompt'])

    tokenized_batch['input_ids'] = tokenized_batch['input_ids'].view(batch_size, num_choices, -1)
    tokenized_batch['attention_mask'] = tokenized_batch['attention_mask'].view(batch_size, num_choices, -1)

    # Convert labels to a tensor
    tokenized_batch['labels'] = torch.tensor(all_labels)

    return tokenized_batch

# Create Hugging Face Dataset from pandas DataFrame
hf_dataset = Dataset.from_pandas(dataset_df)

# Apply the preprocessing function
tokenized_hf_dataset = hf_dataset.map(preprocess_function, batched=True, remove_columns=dataset_df.columns.tolist())

# Inspect the first item of the tokenized dataset
first_item = tokenized_hf_dataset[0]
# Explicitly convert to tensor to get shape, as set_format('torch') might be problematic
input_ids_tensor = torch.tensor(first_item['input_ids'])

print(f"Shape of input_ids for the first dataset item: {input_ids_tensor.shape}")
print(f"Number of tokenized choices stored in input_ids: {input_ids_tensor.shape[0]}")

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Shape of input_ids for the first dataset item: torch.Size([5, 128])
Number of tokenized choices stored in input_ids: 5


# Question 9

In [10]:
from datasets import Dataset
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
import numpy as np

# Create a dataset for the first 32 rows
dataset_df_q9 = train_df.head(32)
hf_dataset_q9 = Dataset.from_pandas(dataset_df_q9)

# Apply the preprocessing function defined earlier
tokenized_hf_dataset_q9 = hf_dataset_q9.map(preprocess_function, batched=True, remove_columns=dataset_df_q9.columns.tolist())

# Define training arguments
training_args = TrainingArguments(
    output_dir="./lora_training_output",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    num_train_epochs=1, # We will control with max_steps
    max_steps=4,
    weight_decay=0.01,
    logging_dir="./lora_logs",
    logging_steps=1,
    save_steps=100,
    report_to=["none"],
    remove_unused_columns=False,
)

# Data collator for padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Initialize Trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_hf_dataset_q9,
    data_collator=data_collator,
)

# Train the model
train_results = trainer.train()

# Get the final global_step
final_global_step = train_results.global_step
print(f"The final global_step is: {final_global_step}")

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,1.698888
2,1.587914
3,1.724996
4,1.577098


The final global_step is: 4


# Question 10

In [11]:
import torch.nn.functional as F

peft_model.eval()

with torch.no_grad():
    inference_outputs = peft_model(input_ids=input_ids_reshaped)
    logits_q10 = inference_outputs.logits

# Apply softmax to get probabilities
probabilities = F.softmax(logits_q10, dim=-1)

# Option E corresponds to the last logit (index 4)
probability_option_e = probabilities[0, 4].item()

print(f"The probability assigned to Option E is: {probability_option_e:.4f}")


The probability assigned to Option E is: 0.2065
